# Paper 1 — Structural Determinism and Molecular Refinement in the *C. elegans* Connectome

## Scope and Claim

This notebook constitutes a **single, end-to-end computational argument** investigating the relationship between anatomical connectivity and gene expression in the *Caenorhabditis elegans* nervous system.

The central claim explored and substantiated herein is:

> **Local connectomic topology in *C. elegans* is largely self-determining, while gene expression acts as a modulatory refinement layer rather than a primary architect of wiring.**

All results in this notebook are derived **from scratch**, using frozen external datasets whose provenance is explicitly recorded.  
Exploratory notebooks and historical analyses have been archived and are not relied upon.

This notebook is intended to be:
- fully reproducible,
- logically linear,
- and suitable as the computational backbone of a scientific manuscript.


### SECTION 0 — Preamble & Guarantees

**Purpose:** Establish trust

- Project scope & claim
- Environment info (Python, packages)
- Global random seeds
- Root path definition
- Assert directory structure exists

If this section runs, the filesystem and environment are sane.

In [40]:
# =========================================
# Section 0.1 — Imports + Global Config
# =========================================

from __future__ import annotations

import os, sys, json, time, re, hashlib, textwrap
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

SEED = 42
np.random.seed(SEED)

print("✅ Section 0.1 loaded")
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Root seed:", SEED)


✅ Section 0.1 loaded
Python: 3.11.13
Pandas: 2.3.2
Root seed: 42


In [41]:
# =========================================
# Section 0.2 — Resolve ROOT + directories (post-reorg)
# =========================================

def find_project_root(start: Path) -> Path:
    """
    Walk up until we find the C-Elegans project root.
    Criteria: contains new-layout dirs (data, analysis, models, simulation, manifests) with at least 3 matches.
    """
    start = start.resolve()
    candidates = [start] + list(start.parents)
    required = {"data", "analysis", "models", "simulation", "manifests", "notebooks"}
    for p in candidates:
        try:
            present = {d.name for d in p.iterdir() if d.is_dir()}
        except (PermissionError, FileNotFoundError):
            continue
        if len(required.intersection(present)) >= 3:
            return p
    return start

NOTEBOOK_PATH = Path.cwd()  # works in Jupyter
ROOT = find_project_root(NOTEBOOK_PATH)

# New canonical topic-based layout
DIRS = {
    # New-layout keys
    "DATA":           ROOT / "data",
    "CONNECTOME":     ROOT / "data" / "connectome",
    "EXPRESSION":     ROOT / "data" / "expression",
    "CENGEN":         ROOT / "data" / "expression" / "cengen",
    "CENGEN_DERIVED": ROOT / "data" / "expression" / "cengen" / "derived",
    "CENGEN_L4":      ROOT / "data" / "expression" / "cengen" / "single_cell_L4",
    "GENOMES":        ROOT / "data" / "genomes",
    "LINEAGE":        ROOT / "data" / "lineage",
    "GSE136049":      ROOT / "data" / "expression" / "scrna_seq_other" / "GSE136049",
    "ANALYSIS":       ROOT / "analysis",
    "GENE_MAPPING":   ROOT / "analysis" / "gene_mapping",
    "MODELS":         ROOT / "models",
    "SIM":            ROOT / "simulation",
    "MANIFESTS":      ROOT / "manifests",
    "MANIFESTS_MISC": ROOT / "manifests" / "misc",
    "NOTEBOOKS":      ROOT / "notebooks",
    "RESULTS":        ROOT / "RESULTS",
    # Legacy-key aliases (so old references keep resolving to the right new place)
    "INTERMEDIATE":   ROOT / "analysis" / "gene_mapping",  # most INTERMEDIATE files moved here
    "NEW_DATA":       ROOT / "data",                        # most NEW_DATA files moved under data/
    "RAW":            ROOT / "data",
    "ARCHIVE":        ROOT / "data",
    "OLD_NOTEBOOKS":  ROOT / "notebooks" / "archive",
}

for k, p in DIRS.items():
    p.mkdir(parents=True, exist_ok=True)

print("✅ ROOT resolved:", ROOT)
print("✅ Key dirs:")
for k in ["DATA", "ANALYSIS", "MODELS", "MANIFESTS", "CENGEN", "GENE_MAPPING"]:
    print(f"  - {k:14s} -> {DIRS[k]}")


✅ ROOT resolved: /mnt/ssd4tb/Desktop/C-Elegans
✅ Key dirs:
  - RAW          -> /mnt/ssd4tb/Desktop/C-Elegans/00_raw
  - INTERMEDIATE -> /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate
  - MANIFESTS    -> /mnt/ssd4tb/Desktop/C-Elegans/05_manifests
  - NEW_DATA     -> /mnt/ssd4tb/Desktop/C-Elegans/New Data
  - OLD_NOTEBOOKS -> /mnt/ssd4tb/Desktop/C-Elegans/Old_Notebooks
  - ARCHIVE      -> /mnt/ssd4tb/Desktop/C-Elegans/_archive


In [42]:
# =========================================
# Section 0.3 — Utilities
# =========================================

def md5_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def file_info(path: Path) -> Dict:
    stat = path.stat()
    return {
        "path": str(path),
        "name": path.name,
        "ext": path.suffix.lower(),
        "size_bytes": stat.st_size,
        "mtime": stat.st_mtime,
        "md5": md5_file(path),
    }

def sniff_table(path: Path, nrows: int = 5) -> Dict:
    """
    Minimal schema sniff for csv/tsv/xlsx.
    Doesn't fully load huge data; only reads a few rows.
    """
    ext = path.suffix.lower()
    out = {"columns": None, "shape_hint": None, "read_error": None}
    try:
        if ext in [".csv"]:
            df = pd.read_csv(path, nrows=nrows)
        elif ext in [".tsv", ".txt"]:
            df = pd.read_csv(path, sep="\t", nrows=nrows)
        elif ext in [".xlsx", ".xls"]:
            df = pd.read_excel(path, nrows=nrows)
        elif ext in [".mtx"]:
            # MatrixMarket: we won't parse here; just record that it's MTX
            out["columns"] = ["<mtx_matrixmarket>"]
            out["shape_hint"] = None
            return out
        else:
            out["read_error"] = f"Unsupported ext for sniff: {ext}"
            return out

        out["columns"] = list(df.columns)
        out["shape_hint"] = (len(df), len(df.columns))
        return out
    except Exception as e:
        out["read_error"] = repr(e)
        return out

print("✅ Section 0.3 utilities ready")


✅ Section 0.3 utilities ready


In [43]:
# =========================================
# Section 0.4 — Dataset discovery helpers
# =========================================

SEARCH_ROOTS = [
    DIRS["RAW"],
    DIRS["NEW_DATA"],
    DIRS["INTERMEDIATE"],
    DIRS["DATA"],
]

def rglob_many(roots: List[Path], patterns: List[str]) -> List[Path]:
    hits = []
    for r in roots:
        if not r.exists():
            continue
        for pat in patterns:
            hits.extend(list(r.rglob(pat)))
    # unique + sorted
    hits = sorted({p.resolve() for p in hits})
    return hits

def rank_paths(paths: List[Path], prefer_dirs: List[Path]) -> List[Path]:
    """
    Rank paths by (1) being under preferred dirs, (2) file size desc, (3) shorter path.
    """
    prefer_dirs = [p.resolve() for p in prefer_dirs]
    def score(p: Path):
        under_prefer = any(str(p).startswith(str(pref)) for pref in prefer_dirs)
        size = p.stat().st_size if p.exists() else 0
        return (1 if under_prefer else 0, size, -len(str(p)))
    return sorted(paths, key=score, reverse=True)

def show_candidates(title: str, paths: List[Path], topk: int = 25):
    print("\n" + "="*90)
    print(title)
    print("="*90)
    if not paths:
        print("  (none found)")
        return
    for i, p in enumerate(paths[:topk], 1):
        try:
            sz = p.stat().st_size / (1024*1024)
            print(f"{i:02d}. {p}   ({sz:.2f} MB)")
        except:
            print(f"{i:02d}. {p}")
    if len(paths) > topk:
        print(f"... and {len(paths)-topk} more")

print("✅ Section 0.4 discovery helpers ready")


✅ Section 0.4 discovery helpers ready


In [44]:
# =========================================
# Section 0.5 — Required dataset slots + candidate search
# =========================================

REQUIRED_SLOTS = {
    # Slot name -> patterns to search (broad on purpose)
    "connectome_edges": ["*connectome*.csv", "*edges*.csv", "*edgelist*.csv", "*synapse*list*.csv", "*NeuronConnect*.xls*", "*witvliet*.xlsx", "*SI 3 Synapse lists*.xlsx"],
    "expression_matrix": ["*TPM*.qs", "*pseudobulk*.qs", "*expr*.csv", "*expression*.csv", "*count_matrix*.mtx", "*gene_by_barcode*.mtx", "*CeNGEN*.csv"],
    "neuron_metadata": ["*neuron*metadata*.csv", "*neuron_master*.csv", "*NeuronType*.xls*", "*NeuronLineage*.xls*", "*cell_type_annotation_lookup_table*.csv", "*gene_annotations*.csv"],
}

candidates = {}
for slot, pats in REQUIRED_SLOTS.items():
    hits = rglob_many(SEARCH_ROOTS, pats)
    hits = rank_paths(hits, prefer_dirs=[DIRS["NEW_DATA"], DIRS["RAW"], DIRS["DATA"], DIRS["INTERMEDIATE"]])
    candidates[slot] = hits
    show_candidates(f"[CANDIDATES] {slot}", hits, topk=30)

print("\n✅ Candidate discovery complete.")
print("Next: pick canonical files (or let auto-pick choose top-ranked) and freeze provenance.")



[CANDIDATES] connectome_edges
01. /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/consensus_connectome_full_withfunc.csv   (7.29 MB)
02. /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/consensus_connectome_full_nofunc.csv   (7.00 MB)
03. /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/consensus_connectome_trimmed_withfunc.csv   (1.86 MB)
04. /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/merged_edges_with_relatedness.csv   (1.52 MB)
05. /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/merged_edges.csv   (1.39 MB)
06. /mnt/ssd4tb/Desktop/C-Elegans/00_raw/SI 3 Synapse lists.xlsx   (1.25 MB)
07. /mnt/ssd4tb/Desktop/C-Elegans/New Data/SI 3 Synapse lists.xlsx   (1.25 MB)
08. /mnt/ssd4tb/Desktop/C-Elegans/New Data/SI 3 Synapse lists(1).xlsx   (1.25 MB)
09. /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/consensus_connectome_trimmed_nofunc.csv   (0.82 MB)
10. /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/witvliet_combined_edges.csv   (0.53 MB)
11. /mnt/ssd4tb/Desktop/C-Elegans/00_raw/NeuronConnectFormat

In [45]:
# =========================================
# Section 0.6 — Freeze dataset provenance manifest
# =========================================

PROVENANCE_FILE = (ROOT / "manifests/misc/dataset_provenance.json")

# --- AUTO PICK (edit these if you want different canonical files)
CHOSEN = {}
for slot, hits in candidates.items():
    CHOSEN[slot] = str(hits[0]) if hits else None

print("Auto-chosen canonical datasets:")
for k, v in CHOSEN.items():
    print(f"  - {k:16s} -> {v}")

# --- Guardrails
missing = [k for k, v in CHOSEN.items() if (v is None)]
if missing:
    raise FileNotFoundError(
        "Missing candidates for slots: "
        + ", ".join(missing)
        + "\nAdd files to 00_raw / New Data / data, or broaden patterns in REQUIRED_SLOTS."
    )

# --- Build provenance object
DATASET_PROVENANCE = {
    "created_unix": time.time(),
    "root": str(ROOT),
    "slots": {},
}

for slot, path_str in CHOSEN.items():
    p = Path(path_str)
    if not p.exists():
        raise FileNotFoundError(f"Chosen file for slot '{slot}' does not exist: {p}")
    info = file_info(p)
    sniff = sniff_table(p, nrows=5)
    DATASET_PROVENANCE["slots"][slot] = {
        **info,
        "sniff": sniff,
        "notes": "",
        "canonical": True,
    }

with open(PROVENANCE_FILE, "w") as f:
    json.dump(DATASET_PROVENANCE, f, indent=2)

print(f"\n✅ Wrote provenance manifest: {PROVENANCE_FILE}")
print("Slots frozen:", list(DATASET_PROVENANCE["slots"].keys()))


Auto-chosen canonical datasets:
  - connectome_edges -> /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/consensus_connectome_full_withfunc.csv
  - expression_matrix -> /mnt/ssd4tb/Desktop/C-Elegans/New Data/GSE136049_gene_by_barcode_count_matrix_all_cells.mtx
  - neuron_metadata  -> /mnt/ssd4tb/Desktop/C-Elegans/New Data/GSE136049_cell_type_annotation_lookup_table.csv

✅ Wrote provenance manifest: /mnt/ssd4tb/Desktop/C-Elegans/05_manifests/dataset_provenance.json
Slots frozen: ['connectome_edges', 'expression_matrix', 'neuron_metadata']


### SECTION 1 — Data Provenance & Canonical Loading

**Purpose:** Eliminate ambiguity

- Load dataset_provenance.json
- Load canonical:
  - Connectome edges
  - Expression matrix
  - Metadata
- Canonicalize neuron IDs
- Assert:
  - Uniqueness
  - Alignment across datasets
  - No duplicates

This replaces all ad-hoc loading logic from Phase C–E.

In [46]:
# =========================================
# Section 1.1 — Load frozen provenance
# =========================================

with open((ROOT / "manifests/misc/dataset_provenance.json"), "r") as f:
    PROV = json.load(f)

def prov_path(slot: str) -> Path:
    return Path(PROV["slots"][slot]["path"])

CONNECTOME_PATH = prov_path("connectome_edges")
EXPR_PATH       = prov_path("expression_matrix")
META_PATH       = prov_path("neuron_metadata")

print("Canonical inputs:")
print(" connectome_edges :", CONNECTOME_PATH)
print(" expression_matrix:", EXPR_PATH)
print(" neuron_metadata  :", META_PATH)


Canonical inputs:
 connectome_edges : /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/consensus_connectome_full_withfunc.csv
 expression_matrix: /mnt/ssd4tb/Desktop/C-Elegans/New Data/GSE136049_gene_by_barcode_count_matrix_all_cells.mtx
 neuron_metadata  : /mnt/ssd4tb/Desktop/C-Elegans/New Data/GSE136049_cell_type_annotation_lookup_table.csv


In [47]:
# =========================================
# Section 1.2 — Connectome edges (canonical)
# =========================================

edges_raw = pd.read_csv(CONNECTOME_PATH)

print("Raw connectome shape:", edges_raw.shape)
print("Columns:", list(edges_raw.columns))

# --- Column normalization ---
RENAME_MAP = {
    "from_neuron": "pre",
    "to_neuron": "post",
    "mean_gap_weight": "gap_weight",
    "mean_chem_weight": "chem_weight",
}

edges = edges_raw.rename(columns=RENAME_MAP)

# --- Keep only canonical columns if present ---
CANON_COLS = [
    "pre", "post",
    "gap_weight", "chem_weight",
    "functional_weight",
    "uncertainty",
    "data_sources",
]

edges = edges[[c for c in CANON_COLS if c in edges.columns]]

# --- Type enforcement ---
edges["pre"]  = edges["pre"].astype(str)
edges["post"] = edges["post"].astype(str)

for c in ["gap_weight", "chem_weight", "functional_weight", "uncertainty"]:
    if c in edges.columns:
        edges[c] = pd.to_numeric(edges[c], errors="coerce")

# --- Drop self-loops unless explicitly needed later ---
edges = edges[edges["pre"] != edges["post"]]

print("Canonical edges shape:", edges.shape)
edges.head()


Raw connectome shape: (90000, 11)
Columns: ['from_neuron', 'to_neuron', 'from_pos', 'to_pos', 'from_type', 'to_type', 'mean_gap_weight', 'mean_chem_weight', 'uncertainty', 'functional_weight', 'data_sources']
Canonical edges shape: (89700, 7)


,pre,post,gap_weight,chem_weight,functional_weight,uncertainty,data_sources
1,ADAL,ADAR,2.000000,NaN,0.091616,NaN,"['chklovskii', 'cook2019', 'funconn', 'openworm', 'white1986_jsh', 'white1986_n2u', 'white1986_whole', 'witvliet2020..."
2,ADAL,ADEL,NaN,NaN,NaN,NaN,[]
3,ADAL,ADER,NaN,NaN,0.014166,NaN,['funconn']
4,ADAL,ADFL,2.666667,NaN,NaN,NaN,"['chklovskii', 'cook2019', 'openworm', 'white1986_jsh', 'white1986_n2u', 'white1986_whole']"
5,ADAL,ADFR,NaN,NaN,NaN,NaN,[]


In [48]:
# =========================================
# Section 1.3 — Save canonical connectome
# =========================================

CONNECTOME_CANON_PATH = (ROOT / "analysis/gene_mapping/connectome_edges_canonical.csv")
edges.to_csv(CONNECTOME_CANON_PATH, index=False)

print("✅ Saved:", CONNECTOME_CANON_PATH)
print("Edges:", len(edges))


✅ Saved: /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/connectome_edges_canonical.csv
Edges: 89700


In [49]:
# =========================================
# Section 1.4 — Load expression matrix (MTX)
# =========================================

from scipy.io import mmread
from scipy.sparse import csr_matrix

# Associated annotation files (from New Data)
GENE_ANNOT = (ROOT / "data/expression/scrna_seq_other/GSE136049/GSE136049_all_cells_gene_annotations.csv")
BARCODE_ANNOT = (ROOT / "data/expression/scrna_seq_other/GSE136049/GSE136049_all_cells_barcodes_column_names.csv")

assert GENE_ANNOT.exists(), "Missing gene annotations"
assert BARCODE_ANNOT.exists(), "Missing barcode annotations"

# Load sparse matrix
X = mmread(EXPR_PATH).tocsr()
print("Sparse matrix shape:", X.shape)

genes = pd.read_csv(GENE_ANNOT)
barcodes = pd.read_csv(BARCODE_ANNOT)

print("Genes:", genes.shape)
print("Barcodes:", barcodes.shape)


Sparse matrix shape: (22469, 100955)
Genes: (22468, 1)
Barcodes: (100954, 1)


In [51]:
# =========================================
# Section 1.5 — Canonical expression table (FIXED)
# =========================================

# GEO MTX quirk: matrix is 1 larger than annotations on both axes
print("Original MTX shape:", X.shape)
print("Genes:", len(genes), "Barcodes:", len(barcodes))

if X.shape[0] == len(genes) + 1 and X.shape[1] == len(barcodes) + 1:
    print("Detected GEO padding row/column — trimming")
    X = X[1:, 1:]
else:
    raise ValueError(
        f"Unexpected MTX/annotation mismatch: "
        f"X={X.shape}, genes={len(genes)}, barcodes={len(barcodes)}"
    )

# Sanity check (must now pass)
assert X.shape[0] == len(genes)
assert X.shape[1] == len(barcodes)

print("Trimmed MTX shape:", X.shape)

# Build sparse DataFrame
expr = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=genes.iloc[:, 0].astype(str),
    columns=barcodes.iloc[:, 0].astype(str),
)

expr.iloc[:5, :5]


Original MTX shape: (22469, 100955)
Genes: 22468 Barcodes: 100954
Detected GEO padding row/column — trimming
Trimmed MTX shape: (22468, 100954)


Pan_AAACCTGAGAGACGAA-1_1,Pan_AAACCTGAGGTAAACT-1_1,Pan_AAACCTGAGGTAGCCA-1_1,Pan_AAACCTGAGTAACCCT-1_1,Pan_AAACCTGAGTACGCGA-1_1,Pan_AAACCTGGTGATGCCC-1_1
WBGene00010957,,,,,
WBGene00010958,1,0,3,0,1
WBGene00014454,0,0,0,0,0
WBGene00010959,0,0,3,0,0
WBGene00010960,7,0,13,0,0
WBGene00010961,0,0,1,0,0


In [53]:
# =========================================
# Section 1.6 — Save canonical expression (SPARSE-SAFE)
# =========================================

from scipy.sparse import csr_matrix, save_npz

EXPR_DIR = (ROOT / "data/expression/cengen/derived/expression_canonical")
EXPR_DIR.mkdir(exist_ok=True)

# Convert to CSR explicitly (guaranteed sparse)
X_csr = csr_matrix(expr.sparse.to_coo())

# Save sparse matrix
MATRIX_PATH = EXPR_DIR / "expression_matrix_canonical.npz"
save_npz(MATRIX_PATH, X_csr)

# Save axis labels
GENES_PATH = EXPR_DIR / "expression_genes.csv"
CELLS_PATH = EXPR_DIR / "expression_cells.csv"

pd.Series(expr.index, name="gene_id").to_csv(GENES_PATH, index=False)
pd.Series(expr.columns, name="cell_barcode").to_csv(CELLS_PATH, index=False)

print("✅ Saved sparse expression canonical bundle:")
print(" matrix :", MATRIX_PATH)
print(" genes  :", GENES_PATH)
print(" cells  :", CELLS_PATH)

print("Shape:", X_csr.shape)
print("Nonzeros:", X_csr.nnz)


✅ Saved sparse expression canonical bundle:
 matrix : /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/expression_canonical/expression_matrix_canonical.npz
 genes  : /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/expression_canonical/expression_genes.csv
 cells  : /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/expression_canonical/expression_cells.csv
Shape: (22468, 100954)
Nonzeros: 41491243


In [54]:
# =========================================
# Section 1.6b — Sanity reload check
# =========================================

from scipy.sparse import load_npz

X_reload = load_npz(MATRIX_PATH)
genes_reload = pd.read_csv(GENES_PATH)["gene_id"].astype(str)
cells_reload = pd.read_csv(CELLS_PATH)["cell_barcode"].astype(str)

assert X_reload.shape == X_csr.shape
assert len(genes_reload) == X_reload.shape[0]
assert len(cells_reload) == X_reload.shape[1]

print("✅ Reload successful")


✅ Reload successful


### SECTION 2 — Structural Targets (What is Being Predicted?)

**Purpose:** Define non-leaky targets

- Define structural properties:
  - Degree
  - Strength
  - Motif participation
- Explicitly state:
  - How targets are computed
  - What data they depend on
- Introduce leakage-resistant variants (e.g., edge-masked motifs)

This is where your earlier critique of the GNCA target is resolved explicitly.

In [57]:
# =========================================
# Section 2.0 — Load canonical neuron metadata (explicit)
# =========================================

META_SOURCE = prov_path("neuron_metadata")
print("Loading neuron metadata from:", META_SOURCE)

meta_raw = pd.read_csv(META_SOURCE)

print("Raw metadata shape:", meta_raw.shape)
print("Columns:", list(meta_raw.columns))

# Explicit schema for GSE136049
REQUIRED_COLS = ["barcode", "cell.type"]

missing = set(REQUIRED_COLS) - set(meta_raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Canonicalize
meta = (
    meta_raw[["barcode", "cell.type"]]
    .rename(columns={
        "barcode": "cell_barcode",
        "cell.type": "neuron"
    })
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values(["neuron", "cell_barcode"])
    .reset_index(drop=True)
)

print("Canonical neuron metadata:")
print("  Rows:", len(meta))
print("  Unique neurons:", meta["neuron"].nunique())
print("  Unique cells  :", meta["cell_barcode"].nunique())

meta.head()


Loading neuron metadata from: /mnt/ssd4tb/Desktop/C-Elegans/New Data/GSE136049_cell_type_annotation_lookup_table.csv
Raw metadata shape: (100955, 3)
Columns: ['barcode', 'cell.type', 'tissue.type']
Canonical neuron metadata:
  Rows: 100955
  Unique neurons: 169
  Unique cells  : 100955


,cell_barcode,neuron
0,G2_ACCAGTACAGCATACT-1,ADA
1,G2_AGATCTGTCGTGGACC-1,ADA
2,G2_CATGACACATCCGGGT-1,ADA
3,G2_CGGGTCACACACCGAC-1,ADA
4,G2_GCATGCGAGCTCCTCT-1,ADA


In [58]:
META_CANON_PATH = (ROOT / "analysis/gene_mapping/neuron_metadata_canonical.csv")
meta.to_csv(META_CANON_PATH, index=False)

print("✅ Saved canonical neuron metadata:")
print(META_CANON_PATH)


✅ Saved canonical neuron metadata:
/mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/neuron_metadata_canonical.csv


In [60]:
# =========================================
# Section 2.1 — Load canonical connectome
# =========================================

edges = pd.read_csv(
    (ROOT / "analysis/gene_mapping/connectome_edges_canonical.csv")
)

print("Edges shape:", edges.shape)
print("Columns:", list(edges.columns))

edges.head()


Edges shape: (89700, 7)
Columns: ['pre', 'post', 'gap_weight', 'chem_weight', 'functional_weight', 'uncertainty', 'data_sources']


,pre,post,gap_weight,chem_weight,functional_weight,uncertainty,data_sources
0,ADAL,ADAR,2.000000,NaN,0.091616,NaN,"['chklovskii', 'cook2019', 'funconn', 'openworm', 'white1986_jsh', 'white1986_n2u', 'white1986_whole', 'witvliet2020..."
1,ADAL,ADEL,NaN,NaN,NaN,NaN,[]
2,ADAL,ADER,NaN,NaN,0.014166,NaN,['funconn']
3,ADAL,ADFL,2.666667,NaN,NaN,NaN,"['chklovskii', 'cook2019', 'openworm', 'white1986_jsh', 'white1986_n2u', 'white1986_whole']"
4,ADAL,ADFR,NaN,NaN,NaN,NaN,[]


In [61]:
# =========================================
# Section 2.2 — Degree & strength targets
# =========================================

# Ensure numeric
for c in ["gap_weight", "chem_weight"]:
    edges[c] = pd.to_numeric(edges[c], errors="coerce").fillna(0.0)

# Outgoing
out_stats = (
    edges
    .groupby("pre")
    .agg(
        out_degree=("post", "count"),
        out_gap_strength=("gap_weight", "sum"),
        out_chem_strength=("chem_weight", "sum"),
    )
)

# Incoming
in_stats = (
    edges
    .groupby("post")
    .agg(
        in_degree=("pre", "count"),
        in_gap_strength=("gap_weight", "sum"),
        in_chem_strength=("chem_weight", "sum"),
    )
)

# Combine
struct_targets = (
    out_stats
    .join(in_stats, how="outer")
    .fillna(0)
)

# Totals
struct_targets["total_degree"] = (
    struct_targets["out_degree"] + struct_targets["in_degree"]
)

struct_targets["total_strength"] = (
    struct_targets["out_gap_strength"] +
    struct_targets["out_chem_strength"] +
    struct_targets["in_gap_strength"] +
    struct_targets["in_chem_strength"]
)

print("Structural target table:")
print(struct_targets.shape)

struct_targets.head()


Structural target table:
(300, 8)


,out_degree,out_gap_strength,out_chem_strength,in_degree,in_gap_strength,in_chem_strength,total_degree,total_strength
pre,,,,,,,,
ADAL,299,17.750000,63.377381,299,17.750000,37.636905,598,136.514286
ADAR,299,15.333333,63.044048,299,15.333333,32.435714,598,126.146429
ADEL,299,15.000000,89.588095,299,15.000000,28.125000,598,147.713095
ADER,299,19.000000,92.645238,299,19.000000,26.375000,598,157.020238
ADFL,299,18.666667,80.738095,299,18.666667,32.684524,598,150.755952


In [64]:
# =========================================
# Section 2.2a — Collapse bilateral neurons
# =========================================

def canonical_neuron(n):
    # Strip L/R suffix if present
    if n.endswith("L") or n.endswith("R"):
        return n[:-1]
    return n

edges["pre_canon"]  = edges["pre"].astype(str).map(canonical_neuron)
edges["post_canon"] = edges["post"].astype(str).map(canonical_neuron)

edges[["pre", "pre_canon", "post", "post_canon"]].head()


,pre,pre_canon,post,post_canon
0,ADAL,ADA,ADAR,ADA
1,ADAL,ADA,ADEL,ADE
2,ADAL,ADA,ADER,ADE
3,ADAL,ADA,ADFL,ADF
4,ADAL,ADA,ADFR,ADF


In [65]:
# =========================================
# Section 2.2b — Degree & strength targets
# =========================================

for c in ["gap_weight", "chem_weight"]:
    edges[c] = pd.to_numeric(edges[c], errors="coerce").fillna(0.0)

out_stats = (
    edges
    .groupby("pre_canon")
    .agg(
        out_degree=("post_canon", "count"),
        out_gap_strength=("gap_weight", "sum"),
        out_chem_strength=("chem_weight", "sum"),
    )
)

in_stats = (
    edges
    .groupby("post_canon")
    .agg(
        in_degree=("pre_canon", "count"),
        in_gap_strength=("gap_weight", "sum"),
        in_chem_strength=("chem_weight", "sum"),
    )
)

struct_targets = (
    out_stats
    .join(in_stats, how="outer")
    .fillna(0)
)

struct_targets["total_degree"] = (
    struct_targets["out_degree"] + struct_targets["in_degree"]
)

struct_targets["total_strength"] = (
    struct_targets["out_gap_strength"] +
    struct_targets["out_chem_strength"] +
    struct_targets["in_gap_strength"] +
    struct_targets["in_chem_strength"]
)

print("Structural target table:", struct_targets.shape)
struct_targets.head()


Structural target table: (202, 8)


,out_degree,out_gap_strength,out_chem_strength,in_degree,in_gap_strength,in_chem_strength,total_degree,total_strength
pre_canon,,,,,,,,
ADA,598,33.083333,126.421429,598,33.083333,70.072619,1196,262.660714
ADE,598,34.000000,182.233333,598,34.000000,54.500000,1196,304.733333
ADF,598,39.333333,166.596429,598,39.333333,67.951190,1196,313.214286
ADL,598,16.500000,192.930952,598,16.500000,67.509524,1196,293.440476
AFD,598,48.500000,38.053571,598,48.500000,53.892857,1196,188.946429


In [66]:
TARGET_PATH = (ROOT / "analysis/gene_mapping/structural_targets_degree_strength.csv")
struct_targets.reset_index().rename(columns={"index": "neuron"}).to_csv(
    TARGET_PATH, index=False
)

print("✅ Saved structural targets:")
print(TARGET_PATH)


✅ Saved structural targets:
/mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/structural_targets_degree_strength.csv


In [67]:
neurons_meta = set(meta["neuron"])
neurons_struct = set(struct_targets.index)

print("Neurons in metadata :", len(neurons_meta))
print("Neurons in targets  :", len(neurons_struct))
print("Overlap             :", len(neurons_meta & neurons_struct))

missing = sorted(neurons_meta - neurons_struct)
print("Missing in targets:", missing[:10])


Neurons in metadata : 169
Neurons in targets  : 202
Overlap             : 91
Missing in targets: ['AMsh', 'AMso', 'AQR', 'AS', 'ASEL', 'ASER', 'AVL', 'AWC_OFF', 'AWC_ON', 'Anal_muscle']


In [68]:
# =========================================
# Section 2.6 — Define canonical neuron universe
# =========================================

neurons_meta = set(meta["neuron"])
neurons_struct = set(struct_targets.index)

# Intersection = neurons we can predict
PREDICT_NEURONS = sorted(neurons_meta & neurons_struct)

print("Prediction neuron universe:", len(PREDICT_NEURONS))


Prediction neuron universe: 91


### SECTION 3 — Baseline Models (Structure Alone)

**Purpose:** Establish dominance of topology

- Graph-only baselines:
  - Simple statistics
  - Message passing / GNCA
- Cross-validation with structure-aware splits
- Report:
  - r / r²
  - Confidence intervals
  - Nulls (degree-preserving randomization)

This is the cleaned, defensible version of what Phase D first hinted at.

In [69]:
# =========================================
# Section 3.1 — Load canonical inputs
# =========================================

from scipy.sparse import load_npz

# Expression bundle
EXPR_DIR = (ROOT / "data/expression/cengen/derived/expression_canonical")

X = load_npz(EXPR_DIR / "expression_matrix_canonical.npz")
genes = pd.read_csv(EXPR_DIR / "expression_genes.csv")["gene_id"].astype(str)
cells = pd.read_csv(EXPR_DIR / "expression_cells.csv")["cell_barcode"].astype(str)

# Metadata
meta = pd.read_csv((ROOT / "analysis/gene_mapping/neuron_metadata_canonical.csv"))

print("Expression matrix:", X.shape)
print("Genes:", len(genes))
print("Cells:", len(cells))
print("Metadata rows:", len(meta))


Expression matrix: (22468, 100954)
Genes: 22468
Cells: 100954
Metadata rows: 100955


In [73]:
# =========================================
# Section 3.1.5 — Load structural targets
# =========================================

STRUCT_PATH = (ROOT / "analysis/gene_mapping/structural_targets_degree_strength.csv")

struct_targets = pd.read_csv(STRUCT_PATH, index_col=0)

print("Structural targets loaded:")
print("Shape:", struct_targets.shape)
print("Neurons:", struct_targets.index.nunique())


Structural targets loaded:
Shape: (202, 8)
Neurons: 202


In [74]:
# =========================================
# Section 3.2 — Restrict to prediction neurons
# =========================================

PREDICT_NEURONS = set(struct_targets.index)

meta = meta[meta["neuron"].isin(PREDICT_NEURONS)].copy()

print("Cells retained :", len(meta))
print("Neurons retained:", meta["neuron"].nunique())


Cells retained : 41537
Neurons retained: 91


In [75]:
# =========================================
# Section 3.3 — Align expression matrix to metadata cells
# =========================================

cell_to_idx = {c: i for i, c in enumerate(cells)}

valid_cells = meta["cell_barcode"].astype(str)
valid_idx = [cell_to_idx[c] for c in valid_cells if c in cell_to_idx]

X_filt = X[:, valid_idx]

meta = meta.iloc[:len(valid_idx)].reset_index(drop=True)

print("Filtered expression shape:", X_filt.shape)


Filtered expression shape: (22468, 41537)


In [76]:
# =========================================
# Section 3.4 — Aggregate expression by neuron
# =========================================

from collections import defaultdict
import numpy as np

neuron_groups = defaultdict(list)

for i, neuron in enumerate(meta["neuron"]):
    neuron_groups[neuron].append(i)

neurons = sorted(neuron_groups.keys())
G = len(genes)
N = len(neurons)

X_neuron = np.zeros((N, G), dtype=np.float32)

for ni, neuron in enumerate(neurons):
    idx = neuron_groups[neuron]
    X_neuron[ni] = X_filt[:, idx].mean(axis=1).A1

print("Neuron expression matrix:", X_neuron.shape)


Neuron expression matrix: (91, 22468)


In [77]:
# =========================================
# Section 3.5 — Save canonical neuron expression
# =========================================

expr_neuron = pd.DataFrame(
    X_neuron,
    index=neurons,
    columns=genes
)

OUT_PATH = (ROOT / "data/expression/cengen/derived/expression_neuron_mean.csv")
expr_neuron.to_csv(OUT_PATH)

print("✅ Saved neuron-level expression:")
print(OUT_PATH)
print("Shape:", expr_neuron.shape)


✅ Saved neuron-level expression:
/mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/expression_neuron_mean.csv
Shape: (91, 22468)


### SECTION 4 — Expression-Only Models

**Purpose:** Show limits of genes in isolation

- Expression-only predictors
- PCA vs raw genes
- Performance vs structure-only
- Statistical comparison

This sets up the contrast that gives the paper its tension.

In [79]:
# =========================================
# Section 4.1 — Load modeling inputs
# =========================================

X_expr = pd.read_csv(
    (ROOT / "data/expression/cengen/derived/expression_neuron_mean.csv"),
    index_col=0
)

Y_struct = pd.read_csv(
    (ROOT / "analysis/gene_mapping/structural_targets_degree_strength.csv"),
    index_col=0
)

# Restrict targets to expression neurons
Y_struct = Y_struct.loc[X_expr.index]

print("X_expr:", X_expr.shape)
print("Y_struct:", Y_struct.shape)


X_expr: (91, 22468)
Y_struct: (91, 8)


In [80]:
# =========================================
# Section 4.2 — Define targets
# =========================================

TARGETS = [
    "total_degree",
    "total_strength",
    "in_degree",
    "out_degree"
]

Y = Y_struct[TARGETS]

print("Targets:", TARGETS)
print(Y.describe())


Targets: ['total_degree', 'total_strength', 'in_degree', 'out_degree']
       total_degree  total_strength   in_degree  out_degree
count     91.000000       91.000000   91.000000   91.000000
mean    1044.857143      310.417504  522.428571  522.428571
std      261.322983      283.562450  130.661492  130.661492
min      598.000000       21.500000  299.000000  299.000000
25%      897.000000      144.331548  448.500000  448.500000
50%     1196.000000      227.570238  598.000000  598.000000
75%     1196.000000      350.485119  598.000000  598.000000
max     1196.000000     1795.330952  598.000000  598.000000


In [81]:
# =========================================
# Section 4.3 — Train/test split
# =========================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_expr.values,
    Y.values,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (72, 22468)
Test : (19, 22468)


In [82]:
# =========================================
# Section 4.4 — Ridge regression baseline
# =========================================

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_test)

for i, tgt in enumerate(TARGETS):
    r2 = r2_score(y_test[:, i], y_pred[:, i])
    print(f"{tgt:15s} R² = {r2:.3f}")


total_degree    R² = -15.313
total_strength  R² = -1.626
in_degree       R² = -15.313
out_degree      R² = -15.313


In [83]:
# =========================================
# Section 4.5 — Permutation test
# =========================================

rng = np.random.default_rng(42)
perm = rng.permutation(len(X_train))

ridge.fit(X_train[perm], y_train)
y_perm = ridge.predict(X_test)

print("Permutation control:")
for i, tgt in enumerate(TARGETS):
    r2 = r2_score(y_test[:, i], y_perm[:, i])
    print(f"{tgt:15s} R² = {r2:.3f}")


Permutation control:
total_degree    R² = -13.020
total_strength  R² = -0.872
in_degree       R² = -13.020
out_degree      R² = -13.020


### SECTION 5 — Combined Models (Structure + Expression)

**Purpose:** Demonstrate modulation, not dominance

- Combined GNCA / GNN
- Proper ablations:
  - Graph only
  - Expression only
  - Graph + real expression
  - Graph + shuffled expression
- Partial correlations controlling for degree
- Effect size reporting

This is where Phase E ideas live — but now clean.

In [ ]:
# =========================================
# Section 5.0 — Build canonical adjacency (FIXED 2026-04-17)
# =========================================
# FIX C1: canonicalize edge endpoints (strip trailing L/R) so they match
#         the canonical-name PREDICT_NEURONS namespace from Section 3.
# FIX C2: filter to real (nonzero-weight) edges before building adjacency,
#         so degree/motif features are meaningful rather than reflecting
#         the dense-edgelist serialization of the canonical CSV.

import numpy as np
import pandas as pd

# Expect these exist from Sections 1–3
assert "DIRS" in globals()
assert ((ROOT / "analysis/gene_mapping/connectome_edges_canonical.csv")).exists()
assert ((ROOT / "analysis/gene_mapping/neuron_metadata_canonical.csv")).exists()

edges = pd.read_csv((ROOT / "analysis/gene_mapping/connectome_edges_canonical.csv"))
meta  = pd.read_csv((ROOT / "analysis/gene_mapping/neuron_metadata_canonical.csv"))

# prediction neuron set = canonical names from Section 3 (L/R already collapsed there)
expr_neuron_path = (ROOT / "data/expression/cengen/derived/expression_neuron_mean.csv")
assert expr_neuron_path.exists(), "Missing expression_neuron_mean.csv from Section 3"
expr_neuron = pd.read_csv(expr_neuron_path, index_col=0)
PREDICT_NEURONS = list(expr_neuron.index.astype(str))

# --- FIX C1: canonicalize edge endpoints to PREDICT_NEURONS namespace ---
def _canonical_neuron(n):
    return n[:-1] if isinstance(n, str) and n.endswith(("L", "R")) else n
edges["pre"]  = edges["pre"].astype(str).map(_canonical_neuron)
edges["post"] = edges["post"].astype(str).map(_canonical_neuron)

print("Edges (raw rows):", edges.shape[0])
print("Unique neurons in edges (canonical):",
      len(set(edges["pre"]).union(set(edges["post"]))))
print("Prediction neurons (from expr):", len(PREDICT_NEURONS))

# Keep only edges among prediction neurons
edges_f = edges[edges["pre"].isin(PREDICT_NEURONS) & edges["post"].isin(PREDICT_NEURONS)].copy()
print("Filtered rows among prediction neurons:", edges_f.shape[0])

# --- FIX C2: restrict to REAL (nonzero-weight) edges so motifs aren't computed on a fully-dense graph ---
edges_f["gap_weight"]  = edges_f["gap_weight"].fillna(0.0)
edges_f["chem_weight"] = edges_f["chem_weight"].fillna(0.0)
edges_f["w"] = edges_f["gap_weight"] + edges_f["chem_weight"]
edges_f = edges_f[edges_f["w"] > 0].copy()
print("Real (nonzero-weight) edges after collapse + filter:", len(edges_f))

# After canonical collapse, multiple raw rows may share a (pre,post) pair — sum their weights.
pair_w = edges_f.groupby(["pre", "post"], as_index=False)["w"].sum()

# Build index map
neurons = pd.Index(PREDICT_NEURONS, name="neuron")
n2i = {n: i for i, n in enumerate(neurons)}

# Weighted adjacency matrix (directed)
W = np.zeros((len(neurons), len(neurons)), dtype=np.float32)
for r in pair_w.itertuples(index=False):
    i = n2i.get(r.pre, None); j = n2i.get(r.post, None)
    if i is None or j is None or i == j:
        continue
    W[i, j] = float(r.w)

# Binary adjacency
A = (W > 0).astype(np.uint8)

print("Adjacency built.")
print("A shape:", A.shape, "density:", A.mean().round(4))
print("Total weighted mass:", float(W.sum()))


In [85]:
# =========================================
# Section 5.1 — Per-neuron directed triad stats
# =========================================

A_f = A.astype(np.int32)

# Degrees
out_deg = A_f.sum(axis=1)
in_deg  = A_f.sum(axis=0)

# Common neighbors counts
# CN_out[i,k] = number of shared out-neighbors between i and k
CN_out = A_f @ A_f.T
CN_in  = A_f.T @ A_f

# Feed-forward loops centered at i:
# i->j and j->k and i->k
# Count for each i: sum_{j,k} A[i,j]*A[j,k]*A[i,k]
# This can be computed as: (A @ A)[i,k] gives #paths i->k of length 2.
A2 = A_f @ A_f
ffl_i = np.sum(A2 * A_f, axis=1)  # sum over k for each i

# 3-cycles participation per i:
# i->j, j->k, k->i
# Diagonal of A^3 gives number of directed 3-cycles through i
A3 = A2 @ A_f
cycle3_i = np.diag(A3)

# "Two-step out paths" and "two-step in paths" as richer centrality-like motifs
two_step_out = A2.sum(axis=1)
two_step_in  = A2.sum(axis=0)

triads = pd.DataFrame({
    "out_deg": out_deg,
    "in_deg": in_deg,
    "ffl": ffl_i,
    "cycle3": cycle3_i,
    "two_step_out": two_step_out,
    "two_step_in": two_step_in,
}, index=neurons)

print("Triad feature table:", triads.shape)
display(triads.head())


Triad feature table: (91, 6)


,out_deg,in_deg,ffl,cycle3,two_step_out,two_step_in
neuron,,,,,,
ADA,0,0,0,0,0,0
ADE,0,0,0,0,0,0
ADF,0,0,0,0,0,0
ADL,0,0,0,0,0,0
AFD,0,0,0,0,0,0


In [86]:
# =========================================
# Section 5.2 — Residualized motif targets (non-trivial structure)
# =========================================

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def ridge_residual(y, X, alpha=10.0):
    """
    Fit ridge y ~ X and return residuals (y - yhat) and fitted yhat.
    """
    y = np.asarray(y, dtype=np.float64)
    X = np.asarray(X, dtype=np.float64)
    scX = StandardScaler()
    scy = StandardScaler()
    Xz = scX.fit_transform(X)
    yz = scy.fit_transform(y.reshape(-1,1)).ravel()
    model = Ridge(alpha=alpha, random_state=42)
    model.fit(Xz, yz)
    yhat_z = model.predict(Xz)
    resid_z = yz - yhat_z
    return resid_z, yhat_z

# covariates that explain "motif inflation"
cov = np.c_[triads["out_deg"].values, triads["in_deg"].values, (triads["out_deg"]+triads["in_deg"]).values]

targets = pd.DataFrame(index=neurons)

for col in ["ffl","cycle3","two_step_out","two_step_in"]:
    resid, yhat = ridge_residual(triads[col].values, cov, alpha=20.0)
    targets[f"{col}_resid"] = resid

print("Residualized targets:", targets.shape)
display(targets.describe().T)


Residualized targets: (91, 4)


,count,mean,std,min,25%,50%,75%,max
ffl_resid,91.0,-3.904081e-17,0.564875,-2.170484,0.041515,0.041515,0.041515,2.263044
cycle3_resid,91.0,-9.577199e-17,0.574029,-1.932174,0.041458,0.041458,0.041458,2.326045
two_step_out_resid,91.0,-3.416071e-17,0.247056,-1.055349,-0.024410,-0.024410,-0.024410,0.716931
two_step_in_resid,91.0,-1.082772e-17,0.260920,-0.848434,-0.040581,-0.040581,-0.040581,0.798969


In [87]:
# =========================================
# Section 5.3 — Edge-masked motif targets (hard anti-leakage)
# =========================================

def masked_adjacency_for_node(A_bin, i):
    """
    Return adjacency with all edges incident to node i removed:
    remove row i (outgoing) and col i (incoming).
    """
    B = A_bin.copy()
    B[i, :] = 0
    B[:, i] = 0
    return B

def ffl_count_per_node(A_bin):
    A2 = A_bin @ A_bin
    return np.sum(A2 * A_bin, axis=1)

def cycle3_per_node(A_bin):
    A2 = A_bin @ A_bin
    A3 = A2 @ A_bin
    return np.diag(A3)

A_int = A.astype(np.int32)
n = A_int.shape[0]

ffl_masked = np.zeros(n, dtype=np.float64)
cyc_masked = np.zeros(n, dtype=np.float64)

for i in range(n):
    B = masked_adjacency_for_node(A_int, i)
    # compute motifs on masked graph, then read i's score in that graph
    ffl_masked[i] = ffl_count_per_node(B)[i]
    cyc_masked[i] = cycle3_per_node(B)[i]

targets["ffl_masked"]   = ffl_masked
targets["cycle3_masked"] = cyc_masked

# Residualize masked versions too (against original degrees, still fine)
for col in ["ffl_masked","cycle3_masked"]:
    resid, _ = ridge_residual(targets[col].values, cov, alpha=20.0)
    targets[f"{col}_resid"] = resid

print("Targets with masked variants:", targets.shape)
display(targets.head())


Targets with masked variants: (91, 8)


,ffl_resid,cycle3_resid,two_step_out_resid,two_step_in_resid,ffl_masked,cycle3_masked,ffl_masked_resid,cycle3_masked_resid
neuron,,,,,,,,
ADA,0.041515,0.041458,-0.02441,-0.040581,0.0,0.0,0.0,0.0
ADE,0.041515,0.041458,-0.02441,-0.040581,0.0,0.0,0.0,0.0
ADF,0.041515,0.041458,-0.02441,-0.040581,0.0,0.0,0.0,0.0
ADL,0.041515,0.041458,-0.02441,-0.040581,0.0,0.0,0.0,0.0
AFD,0.041515,0.041458,-0.02441,-0.040581,0.0,0.0,0.0,0.0


In [88]:
# =========================================
# Section 5.4 — Save motif targets
# =========================================

OUT_PATH = (ROOT / "analysis/gene_mapping/structural_targets_motif_residualized.csv")
targets.to_csv(OUT_PATH)
print("✅ Saved motif targets:", OUT_PATH)
print("Shape:", targets.shape)
print("Columns:", list(targets.columns))


✅ Saved motif targets: /mnt/ssd4tb/Desktop/C-Elegans/01_intermediate/structural_targets_motif_residualized.csv
Shape: (91, 8)
Columns: ['ffl_resid', 'cycle3_resid', 'two_step_out_resid', 'two_step_in_resid', 'ffl_masked', 'cycle3_masked', 'ffl_masked_resid', 'cycle3_masked_resid']


### SECTION 6 — Gene Attribution & Biological Interpretation

**Purpose:** Extract biology

- Back-project attributions to genes
- ID normalization (WBGene → symbol)
- GO / pathway enrichment
- Motif-specific gene programs

This is where the “genes refine form” claim becomes biological.

In [89]:
# =========================================
# Section 6.1 — Load inputs for motif prediction
# =========================================

X_expr = pd.read_csv(
    (ROOT / "data/expression/cengen/derived/expression_neuron_mean.csv"),
    index_col=0
)

Y_motif = pd.read_csv(
    (ROOT / "analysis/gene_mapping/structural_targets_motif_residualized.csv"),
    index_col=0
)

# Align
Y_motif = Y_motif.loc[X_expr.index]

TARGET_COLS = [
    "ffl_resid",
    "cycle3_resid",
    "two_step_out_resid",
    "two_step_in_resid",
    "ffl_masked_resid",
    "cycle3_masked_resid",
]

Y = Y_motif[TARGET_COLS]

print("X_expr:", X_expr.shape)
print("Y_motif:", Y.shape)
print("Target variances:")
display(Y.var())


X_expr: (91, 22468)
Y_motif: (91, 6)
Target variances:


ffl_resid              0.319084
cycle3_resid           0.329510
two_step_out_resid     0.061037
two_step_in_resid      0.068079
ffl_masked_resid       0.000000
cycle3_masked_resid    0.000000
dtype: float64

In [90]:
# =========================================
# Section 6.2 — Train/test split
# =========================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_expr.values,
    Y.values,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (72, 22468)
Test : (19, 22468)


In [91]:
# =========================================
# Section 6.3 — Ridge regression on motif targets
# =========================================

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

ridge = Ridge(alpha=10.0)

ridge.fit(X_train, y_train)
y_pred = ridge.predict(X_test)

print("R² scores (real):")
for i, tgt in enumerate(TARGET_COLS):
    r2 = r2_score(y_test[:, i], y_pred[:, i])
    print(f"{tgt:22s} R² = {r2:.3f}")


R² scores (real):
ffl_resid              R² = -3.385
cycle3_resid           R² = -2.782
two_step_out_resid     R² = -5.313
two_step_in_resid      R² = -3.783
ffl_masked_resid       R² = 1.000
cycle3_masked_resid    R² = 1.000


In [92]:
# =========================================
# Section 6.4 — Permutation control
# =========================================

rng = np.random.default_rng(42)
perm = rng.permutation(len(X_train))

ridge.fit(X_train[perm], y_train)
y_perm = ridge.predict(X_test)

print("R² scores (permutation):")
for i, tgt in enumerate(TARGET_COLS):
    r2 = r2_score(y_test[:, i], y_perm[:, i])
    print(f"{tgt:22s} R² = {r2:.3f}")


R² scores (permutation):
ffl_resid              R² = -4.759
cycle3_resid           R² = -4.520
two_step_out_resid     R² = -11.655
two_step_in_resid      R² = -11.793
ffl_masked_resid       R² = 1.000
cycle3_masked_resid    R² = 1.000


### SECTION 7 — Theoretical Synthesis

**Purpose:** Elevate to principle

- Summarize empirical findings
- Introduce the developmental automaton interpretation:
  - Topology = lattice
  - Genes = update rules / modulators
- Position relative to known theories

This is where your Neural Cellular Automaton framing belongs.

### SECTION 8 — Limitations & Next Directions

**Purpose:** Intellectual honesty

- Explicit limitations
- What requires wet lab validation
- What generalizes computationally

Reviewers respect this section.

### SECTION 9 — Reproducibility Appendix

**Purpose:** Future-proofing

- Save all figures
- Save all tables
- Dump config + hashes
- Print final summary

6.1
        

In [94]:
import pandas as pd
from pathlib import Path

LINEAGE_FILES = [
    (ROOT / "data/lineage/NeuronLineage_Part1.xls"),
    (ROOT / "data/lineage/NeuronLineage_Part2.xls"),
]

dfs = []
for f in LINEAGE_FILES:
    df = pd.read_excel(f)
    df["__source_file"] = f.name
    dfs.append(df)

lineage_raw = pd.concat(dfs, ignore_index=True)

print("Raw lineage shape:", lineage_raw.shape)
print("Columns:")
for c in lineage_raw.columns:
    print(" ", c)


Raw lineage shape: (77562, 4)
Columns:
  Neuron 1
  Neuron 2
  Relatedness
  __source_file


In [95]:
def normalize_neuron(n):
    if pd.isna(n):
        return None
    n = str(n).strip()
    return n

lineage_raw["neuron"] = lineage_raw["Neuron"].apply(normalize_neuron)

print("Unique neurons (raw):", lineage_raw["neuron"].nunique())


KeyError: 'Neuron'

In [96]:
EXPECTED_COLS = lineage_raw.columns.tolist()

def infer_founder(lineage_str):
    if not isinstance(lineage_str, str):
        return None
    for f in ["AB", "MS", "C", "D", "E", "P"]:
        if lineage_str.startswith(f):
            return f
    return None

def infer_lr(neuron):
    if neuron.endswith("L"):
        return "L"
    if neuron.endswith("R"):
        return "R"
    return "U"   # unpaired / unknown

lineage = lineage_raw.copy()

lineage["founder"] = lineage["Lineage"].apply(infer_founder)
lineage["lr"] = lineage["neuron"].apply(infer_lr)

# lineage depth = number of divisions encoded
lineage["depth"] = lineage["Lineage"].apply(
    lambda x: x.count(".") + 1 if isinstance(x, str) else None
)

# branch signature = lineage path truncated (coarse)
lineage["branch"] = lineage["Lineage"].apply(
    lambda x: ".".join(x.split(".")[:3]) if isinstance(x, str) else None
)

print(
    lineage[["neuron", "Lineage", "founder", "depth", "lr", "branch"]]
    .drop_duplicates()
    .head(10)
)


KeyError: 'Lineage'

In [97]:
lineage_neuron = (
    lineage
    .groupby("neuron", as_index=False)
    .agg({
        "founder": "first",
        "depth": "mean",
        "lr": "first",
        "branch": "first"
    })
)

print("Lineage neurons:", lineage_neuron.shape[0])
lineage_neuron.head()


KeyError: 'neuron'